# MongoDB Shell 기초 명령어

## 데이터베이스 관리
- `show dbs` - 모든 데이터베이스 목록 조회
- `use <database>` - 특정 데이터베이스 선택
- `db` - 현재 데이터베이스 확인
- `db.dropDatabase()` - 현재 데이터베이스 삭제

## 컬렉션 관리
- `show collections` - 현재 데이터베이스의 모든 컬렉션 조회
- `db.<collection>.drop()` - 특정 컬렉션 삭제
- `db.createCollection("<collection>")` - 새 컬렉션 생성

## CRUD 작업

### Create (생성)
- `db.<collection>.insertOne({...})` - 단일 문서 삽입
- `db.<collection>.insertMany([{...}, {...}])` - 여러 문서 삽입

### Read (조회)
- `db.<collection>.find()` - 모든 문서 조회
- `db.<collection>.find({query})` - 조건에 맞는 문서 조회
- `db.<collection>.findOne({query})` - 첫 번째 문서 조회
- `db.<collection>.countDocuments({query})` - 문서 개수 확인

### Update (수정)
- `db.<collection>.updateOne({query}, {$set: {...}})` - 단일 문서 수정
- `db.<collection>.updateMany({query}, {$set: {...}})` - 여러 문서 수정
- `db.<collection>.replaceOne({query}, {...})` - 문서 전체 교체

### Delete (삭제)
- `db.<collection>.deleteOne({query})` - 단일 문서 삭제
- `db.<collection>.deleteMany({query})` - 여러 문서 삭제

## 쿼리 옵션
- `.sort({field: 1})` - 오름차순 정렬 (1: 오름차순, -1: 내림차순)
- `.limit(n)` - 결과 개수 제한
- `.skip(n)` - 처음 n개 결과 건너뛰기
- `.pretty()` - 포맷된 출력

## 인덱스 관리
- `db.<collection>.createIndex({field: 1})` - 인덱스 생성
- `db.<collection>.getIndexes()` - 인덱스 조회
- `db.<collection>.dropIndex("<indexName>")` - 인덱스 삭제

데이터 삽입 시 자동으로 입력되는 Object ID가 기본 키 역할을 한다. 시퀀스도 필요 없음

내가 원하는 특정 필드를 기본 키로 사용하려면 속성명을 `_id`로 지정하면 된다.

예시 (MongoDB Shell):
```mongo
nov14_product> db.nov14_product.insertOne({_id: "모나미 153", p_price: 500})
{ acknowledged: true, insertedId: '모나미 153' }
nov14_product> db.nov14_product.insertOne({_id: "모나미 153", p_price: 300})
MongoServerError: E11000 duplicate key error collection: nov14_product.nov14_product index: _id_ dup key: { _id: "모나미 153" }
```

- 참고: `_id` 필드는 컬렉션 내에서 고유해야 하므로 동일한 `_id` 값으로 문서를 다시 삽입하면 중복 키 오류가 발생한다.

비교 연산자($gt 등) 설명 및 샘플 쿼리
------------------------------------

주요 비교 연산자:
- `$gt`: 큰값 (greater than)
- `$gte`: 크거나 같은값 (greater than or equal)
- `$lt`: 작은값 (less than)
- `$lte`: 작거나 같은값 (less than or equal)
- `$eq`: 같은값 (equal) — 보통 `{field: value}` 와 동일
- `$ne`: 같지 않은값 (not equal)
- 참고: 멤버십 관련 `$in`, `$nin` 도 자주 사용됨

사용법(기본 형식):
- 필드에 대해 `{ field: { $op: value } }` 형태로 사용
- 숫자, 날짜(ISODate), 문자열(사전식 비교) 등 유형에 따라 동작

샘플 쿼리 (MongoDB Shell):
```
nov14_product> db.nov14_product.find({p_price: {$lte:100000}})
[
  { _id: '모나미 153', p_price: 500 },
  { _id: '제트스트림 0.5', p_price: 2500 },
  { _id: 'Logitech Lift', p_price: 98000 }
]
nov14_product> db.nov14_product.find({p_price: {$gte:200000}})
[
  { _id: 'iPhone 13 mini', p_price: 790000 },
  { _id: 'Airpods Pro', p_price: 265000 }
]
```

논리 연산자를 활용한 복합 조건 검색
```
nov14_product> db.nov14_product.find({$or:[ {p_price: {$lte: 3000}}, {p_price: {$gte: 300000}}]})
[
  { _id: '모나미 153', p_price: 500 },
  { _id: '제트스트림 0.5', p_price: 2500 },
  { _id: 'iPhone 13 mini', p_price: 790000 }
]

nov14_product> db.nov14_product.find({$and:[ {p_price: {$gte: 0}}, {p_price: {$lte: 10000000}}]})
[
  { _id: '모나미 153', p_price: 500 },
  { _id: '제트스트림 0.5', p_price: 2500 },
  { _id: 'iPhone 13 mini', p_price: 790000 },
  { _id: 'Logitech Lift', p_price: 98000 },
  { _id: 'Airpods Pro', p_price: 265000 }
]

```

> 다만 어차피 내용물로 연산을 할 일이 많으면 RDBMS를 쓰는 게 낫다.

.sort() 메서드를 이용한 데이터 정렬
------------------------------------

`.sort()` 메서드는 쿼리 결과를 특정 필드 기준으로 정렬하는 데 사용된다.

기본 문법:

```mongo
db.<collection>.find(query).sort({ field: 1 })        // 오름차순
db.<collection>.find(query).sort({ field: -1 })       // 내림차순
```

여러 필드로 정렬:

```mongo
db.<collection>.find().sort({ price: -1, name: 1 })
```

예시 (MongoDB Shell):

```mongo
db.nov14_product.find().sort({ p_price: 1 }).limit(5)
db.nov14_product.find({ category: 'pen' }).sort({ p_price: -1, _id: 1 })
```

Pymongo 예시:

```python
from pymongo import MongoClient
client = MongoClient()
col = client.mydb.nov14_product
for doc in col.find().sort([('p_price', -1), ('_id', 1)]).limit(5):
    print(doc)
```

정렬 성능 팁:

- 인덱스 사용: 정렬 필드에 적절한 인덱스를 생성하면 메모리 정렬을 피하고 성능이 좋아진다. (예: `db.nov14_product.createIndex({ p_price: 1 })`)
- 복합 정렬: 필터와 정렬을 함께 사용할 때에는 인덱스가 쿼리 패턴과 맞아야 한다. 일반적으로 인덱스 키 순서가 중요하다.
- 메모리 정렬 제한: 큰 결과를 정렬할 경우 MongoDB는 메모리 기반 정렬을 수행할 수 있으며, 집계(aggregation)에서 `allowDiskUse`를 사용하거나 적절한 인덱스를 만들어야 한다.
- 로케일/대소문자: 언어별 정렬이 필요하면 `collation` 옵션을 사용한다. (예: `db.collection.find().sort({name:1}).collation({locale:'ko'})`)

실행 계획 확인:

```mongo
db.nov14_product.find().sort({ p_price: -1 }).limit(5).explain('executionStats')
```

요약:

- `.sort()`는 다양한 쿼리와 조합해 사용할 수 있으며, 성능을 위해 인덱스 설계가 중요하다.
- 작은 샘플을 확인할 때는 `limit()`와 함께 사용해 빠르게 결과를 확인하자.


skip을 활용해 맨 앞 데이터 n개 '건너뛰고' 조회하기.
```
nov14_product> db.nov14_product.find().sort({_id:1})
[
  { _id: 'Airpods Pro', p_price: 265000 },
  { _id: 'Logitech Lift', p_price: 98000 },
  { _id: 'iPhone 13 mini', p_price: 790000 },
  { _id: '모나미 153', p_price: 500 },
  { _id: '제트스트림 0.5', p_price: 2500 }
]
nov14_product> db.nov14_product.find().sort({_id:1}).skip(1)
[
  { _id: 'Logitech Lift', p_price: 98000 },
  { _id: 'iPhone 13 mini', p_price: 790000 },
  { _id: '모나미 153', p_price: 500 },
  { _id: '제트스트림 0.5', p_price: 2500 }
]
```

limit를 활용해 결과를 n개만 출력하기.
```
nov14_product> db.nov14_product.find().sort({_id:1}).skip(2).limit(3)
[
  { _id: 'iPhone 13 mini', p_price: 790000 },
  { _id: '모나미 153', p_price: 500 },
  { _id: '제트스트림 0.5', p_price: 2500 }
]
```

이미 존재하는 데이터의 수정.

set p_price 0 where _id = "모나미 153"과 같은 동작을 하려면,

db.배열이름.updateMany({객체}, {$set: {객체}})

```
nov14_product> db.nov14_product.updateMany(
... {_id:"모나미 153"},
... {$set: {p_price:0}}
... )
{
  acknowledged: true,
  insertedId: null,
  matchedCount: 1,
  modifiedCount: 1,
  upsertedCount: 0
}
```

데이터의 삭제

delete from ...

> db.배열.deleteMany({객체})

```
db.nov14_product.deleteMany({_id:{$regex: }})